In [1]:
# app.py 
import os, re, json
import numpy as np
import pandas as pd
import torch
import gradio as gr
import plotly.graph_objects as go
import plotly.io as pio                   
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer
from captum.attr import LayerIntegratedGradients

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EMOTION_COLUMNS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

EMOTION_ICONS = {
    "admiration": "👏", "amusement": "😂", "anger": "😡", "annoyance": "😒",
    "approval": "👍", "caring": "❤️", "confusion": "😕", "curiosity": "🔎",
    "desire": "✨", "disappointment": "😞", "disapproval": "👎", "disgust": "🤢",
    "embarrassment": "😳", "excitement": "🤩", "fear": "😨", "gratitude": "🙏",
    "grief": "😢", "joy": "😊", "love": "❤️", "nervousness": "😬",
    "optimism": "🌟", "pride": "🏆", "realization": "💡", "relief": "😌",
    "remorse": "😔", "sadness": "😢", "surprise": "😮", "neutral": "😐",
}

MODEL_DIR = "./goemotions_model_v3"
EVAL_DIR = "./eval_outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LIME_NUM_SAMPLES = 100  

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

with open(os.path.join(MODEL_DIR, "metrics.json")) as f:
    TRAIN_META = json.load(f)
MAX_LEN = TRAIN_META["max_len"]


def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def predict_proba(texts):
    enc = tokenizer(list(texts), truncation=True, padding=True,
                     max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.sigmoid(logits).cpu().numpy()


_lime_explainer = LimeTextExplainer(class_names=EMOTION_COLUMNS)
_embedding_layer = model.distilbert.embeddings
_lig = LayerIntegratedGradients(
    lambda input_ids, attention_mask, target_idx: model(
        input_ids=input_ids, attention_mask=attention_mask
    ).logits[:, target_idx],
    _embedding_layer,
)


def get_lime_word_scores(text, target_idx, num_features=10, num_samples=LIME_NUM_SAMPLES):
    exp = _lime_explainer.explain_instance(
        text, predict_proba, labels=(target_idx,),
        num_features=num_features, num_samples=num_samples
    )
    return exp.as_list(label=target_idx)


def get_ig_word_scores(text, target_idx, steps=32):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    baseline_ids = input_ids.clone()
    baseline_ids[:, 1:-1] = tokenizer.pad_token_id

    attributions = _lig.attribute(
        inputs=input_ids, baselines=baseline_ids,
        additional_forward_args=(attention_mask, target_idx), n_steps=steps,
    ).sum(dim=-1).squeeze(0)
    attributions = attributions / (attributions.norm() + 1e-8)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0).tolist())
    words, scores, cur_word, cur_score = [], [], None, 0.0
    for tok, score in zip(tokens, attributions.tolist()):
        if tok in (tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token):
            continue
        if tok.startswith("##"):
            cur_word += tok[2:]
            cur_score += score
        else:
            if cur_word is not None:
                words.append(cur_word)
                scores.append(cur_score)
            cur_word, cur_score = tok, score
    if cur_word is not None:
        words.append(cur_word)
        scores.append(cur_score)

    pairs = list(zip(words, scores))
    return sorted(pairs, key=lambda x: -abs(x[1]))


def build_highlighted(text, word_score_pairs):
    word_scores = {w.lower(): s for w, s in word_score_pairs}
    matched_keys = set()
    pieces = re.findall(r"\w+|[^\w\s]|\s+", text)
    result = []
    for piece in pieces:
        if piece.strip() == "":
            result.append((piece, None))
            continue
        key = piece.lower()
        score = word_scores.get(key)
        if score is not None:
            matched_keys.add(key)
        else:
            stripped = re.sub(r"[^\w]", "", key)
            score = word_scores.get(stripped)
            if score is not None:
                matched_keys.add(stripped)
        result.append((piece, float(score) if score is not None else None))

    unmatched = [
        w for w in word_scores
        if w not in matched_keys and re.sub(r"[^\w]", "", w) not in matched_keys
    ]
    return result, unmatched


def build_mismatch_note(lime_unmatched, ig_unmatched):
    parts = []
    if lime_unmatched:
        parts.append(f"LIME: {', '.join(lime_unmatched)}")
    if ig_unmatched:
        parts.append(f"Integrated Gradients: {', '.join(ig_unmatched)}")
    if not parts:
        return ""
    return (
        '<p class="mismatch-note">⚠️ Some top-attribution words could not be '
        "matched back onto the original text due to tokenization/punctuation "
        f"differences — {' · '.join(parts)}. Their weight still counted toward "
        "the charts and faithfulness scores below, just not the highlighted "
        "sentence.</p>"
    )


def make_word_bar_chart(word_score_pairs, title, top_k=8):
    pairs = sorted(word_score_pairs, key=lambda x: -abs(x[1]))[:top_k][::-1]
    if not pairs:
        pairs = [("(no words)", 0.0)]
    words = [w for w, _ in pairs]
    scores = [s for _, s in pairs]
    colors = ["#34D399" if s >= 0 else "#F87171" for s in scores]
    patterns = ["" if s >= 0 else "/" for s in scores]
    tags = ["supports" if s >= 0 else "opposes" for s in scores]

    fig = go.Figure(go.Bar(
        x=scores, y=words, orientation="h",
        marker=dict(
            color=colors,
            pattern=dict(shape=patterns, fgcolor="#0B0E1A", size=6),
            line=dict(width=0),
        ),
        customdata=tags,
        hovertemplate="<b>%{y}</b><br>weight: %{x:.3f} (%{customdata})<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=title, font=dict(color="#E2E8F0", size=13)),
        xaxis=dict(title="Attribution weight", color="#CBD5E1",
                   gridcolor="#262B45", zeroline=True, zerolinecolor="#64748B"),
        yaxis=dict(color="#CBD5E1"),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=16, t=40, b=10),
        height=max(220, 40 * len(words) + 80),
        font=dict(color="#CBD5E1"),
    )
    return fig


def make_faithfulness_chart(lime_val, ig_val, title):
    methods = ["LIME", "IG"]
    vals = [lime_val, ig_val]
    base_colors = ["#FBBF24", "#818CF8"]
    patterns = ["" if v >= 0 else "x" for v in vals]

    fig = go.Figure(go.Bar(
        x=methods, y=vals,
        marker=dict(color=base_colors, pattern=dict(shape=patterns, fgcolor="#0B0E1A", size=6)),
        hovertemplate="<b>%{x}</b>: %{y:.3f}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(text=title, font=dict(color="#E2E8F0", size=12)),
        yaxis=dict(color="#CBD5E1", gridcolor="#262B45", zeroline=True, zerolinecolor="#64748B"),
        xaxis=dict(color="#CBD5E1"),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=10, t=40, b=10),
        height=280,
        font=dict(color="#CBD5E1"),
        showlegend=False,
    )
    return fig


def mask_out(text, words):
    if not words:
        return text
    pattern = r'\b(' + '|'.join(re.escape(w) for w in words) + r')\b'
    return re.sub(r'\s+', ' ', re.sub(pattern, '', text, flags=re.IGNORECASE)).strip()


def target_prob(text, target_idx):
    return float(predict_proba([text])[0, target_idx])


def comprehensiveness(text, target_idx, top_words):
    return target_prob(text, target_idx) - target_prob(mask_out(text, top_words), target_idx)


def sufficiency(text, target_idx, top_words):
    kept_only = " ".join(top_words) if top_words else text
    return target_prob(text, target_idx) - target_prob(kept_only, target_idx)


def build_emotion_bars_html(label_dict, top_label):
    rows = []
    for label, prob in label_dict.items():
        if label == top_label:
            continue
        pct = round(prob * 100)
        icon = EMOTION_ICONS.get(label, "")
        rows.append(f"""
        <div class="emo-row">
          <div class="emo-row-top"><span class="emo-name">{icon} {label}</span><span class="emo-pct">{pct}%</span></div>
          <div class="emo-bar-track"><div class="emo-bar-fill" style="width:{pct}%;"></div></div>
        </div>""")
    body = "".join(rows) if rows else '<p class="muted-text">No other labels above threshold.</p>'
    return f'<p class="mini-label">Other detected emotions</p><div class="emo-bars">{body}</div>'


def build_spotlight_html(label, prob):
    icon = EMOTION_ICONS.get(label, "")
    pct = round(prob * 100)
    return f"""
    <div class="spotlight">
      <p class="mini-label">Top prediction</p>
      <div class="spotlight-icon">{icon}</div>
      <div class="spotlight-label">{label.upper()}</div>
      <div class="spotlight-pct">{pct}%</div>
      <div class="spotlight-bar-track"><div class="spotlight-bar-fill" style="width:{pct}%;"></div></div>
    </div>
    """


def build_why_html(label, lime_words, ig_words):
    icon = EMOTION_ICONS.get(label, "")
    combined = list(dict.fromkeys(lime_words[:2] + ig_words[:2]))[:3]
    word_str = ", ".join(f"\u201c{w}\u201d" for w in combined) if combined else "no single dominant word"
    return f"""
    <div class="why-card">
      <p class="mini-label">Why {icon} {label}?</p>
      <p class="why-text">The model's explainers point to {word_str} as the strongest
      evidence in this sentence for this emotion.</p>
    </div>
    """


def build_comparison_html(lime_words, ig_words):
    lime_set = {w.lower() for w in lime_words}
    ig_set = {w.lower() for w in ig_words}
    union = lime_set | ig_set
    agreement = (len(lime_set & ig_set) / len(union)) if union else 0.0

    def word_list_html(words):
        return "".join(f'<li>{w}</li>' for w in words) or "<li class='muted-text'>—</li>"

    return f"""
    <div class="compare-grid">
      <div class="compare-col">
        <p class="mini-label">LIME top words</p>
        <ul class="word-list">{word_list_html(lime_words)}</ul>
      </div>
      <div class="compare-col">
        <p class="mini-label">Integrated Gradients top words</p>
        <ul class="word-list">{word_list_html(ig_words)}</ul>
      </div>
    </div>
    <div class="agreement-row">
      <div class="agreement-bar-track"><div class="agreement-bar-fill" style="width:{agreement*100:.0f}%;"></div></div>
      <span class="agreement-pct">{agreement*100:.0f}%</span>
    </div>
    <p class="summary-text">Word-overlap agreement between LIME and Integrated Gradients —
    a measure of how much the two methods agree with each other, not proof either is
    objectively correct.</p>
    """


def build_history_html(history):
    if not history:
        return '<p class="muted-text">No sentences analyzed yet this session.</p>'
    rows = []
    for item in reversed(history[-6:]):
        icon = EMOTION_ICONS.get(item["label"], "")
        rows.append(f"""
        <div class="history-item">
          <div class="history-top">{icon} <strong>{item['label']}</strong> — {item['pct']}%</div>
          <div class="history-text">"{item['text']}"</div>
        </div>""")
    return "".join(rows)


def build_rank_table_html(rank_df):
    if rank_df is None or rank_df.empty:
        return '<p class="muted-text">No ranked emotions yet.</p>'
    rows = "".join(
        f"<tr><td>{r.Rank}</td><td>{r.Emotion}</td><td>{r.Probability:.3f}</td></tr>"
        for r in rank_df.itertuples()
    )
    return f"""
    <table class="rank-table">
      <thead><tr><th>Rank</th><th>Emotion</th><th>Probability</th></tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """


LEGEND_HTML = """
<div class="legend-row">
  <span class="legend-chip legend-pos">▲ Supports prediction</span>
  <span class="legend-chip legend-neg">▼ Opposes prediction</span>
</div>
<p class="muted-text legend-footnote">Charts below also use a diagonal hatch pattern
on "opposes" bars, so the distinction doesn't rely on color alone.</p>
"""

STATS_HTML = """
<div class="stats-row">
  <div class="stat-card"><div class="stat-num">28</div><div class="stat-label">Emotion labels</div></div>
  <div class="stat-card"><div class="stat-num">2</div><div class="stat-label">Explanation methods</div></div>
  <div class="stat-card"><div class="stat-num">2</div><div class="stat-label">Faithfulness metrics</div></div>
  <div class="stat-card"><div class="stat-num">1</div><div class="stat-label">Unified NLP studio</div></div>
</div>
"""


def analyze(text, threshold, top_n, history, progress=gr.Progress()):
    text = clean_text(text)
    if not text:
        empty_html = build_rank_table_html(pd.DataFrame(columns=["Rank", "Emotion", "Probability"]))
        return ("", "", "", "", "",
                [], None, [], None, None, None,
                "Type a sentence and click **Analyze**.", {}, empty_html,
                build_history_html(history), history)

    progress(0.05, desc="Preparing input")
    probs = predict_proba([text])[0]

    progress(0.2, desc="Ranking emotions")
    ranked = sorted(zip(EMOTION_COLUMNS, probs), key=lambda x: -x[1])
    above_threshold = [(l, float(p)) for l, p in ranked if p >= threshold]
    shown = above_threshold[:int(top_n)] if above_threshold else ranked[:1]
    label_dict = {l: float(p) for l, p in shown}

    target_label = shown[0][0]
    target_idx = EMOTION_COLUMNS.index(target_label)
    spotlight_html = build_spotlight_html(target_label, shown[0][1])
    other_bars_html = build_emotion_bars_html(label_dict, target_label)

    progress(0.35, desc="Computing LIME explanation")
    lime_pairs = get_lime_word_scores(text, target_idx)
    lime_highlight, lime_unmatched = build_highlighted(text, lime_pairs)
    lime_chart = make_word_bar_chart(lime_pairs, f"LIME — top words for '{target_label}'")

    progress(0.65, desc="Computing Integrated Gradients")
    ig_pairs = get_ig_word_scores(text, target_idx)
    ig_highlight, ig_unmatched = build_highlighted(text, ig_pairs)
    ig_chart = make_word_bar_chart(ig_pairs, f"Integrated Gradients — top words for '{target_label}'")

    mismatch_html = build_mismatch_note(lime_unmatched, ig_unmatched)

    lime_words = [w for w, _ in sorted(lime_pairs, key=lambda x: -abs(x[1]))[:3]]
    ig_words = [w for w, _ in sorted(ig_pairs, key=lambda x: -abs(x[1]))[:3]]
    why_html = build_why_html(target_label, lime_words, ig_words)
    comparison_html = build_comparison_html(lime_words, ig_words)

    progress(0.85, desc="Checking faithfulness")
    comp_chart = make_faithfulness_chart(
        comprehensiveness(text, target_idx, lime_words),
        comprehensiveness(text, target_idx, ig_words),
        "Comprehensiveness (higher = better)",
    )
    suff_chart = make_faithfulness_chart(
        sufficiency(text, target_idx, lime_words),
        sufficiency(text, target_idx, ig_words),
        "Sufficiency drop (lower = better)",
    )

    note = (
        f"Explaining **'{target_label}'** — confidence **{shown[0][1]:.2f}**. "
        f"Green pushes *toward* this label, red pushes *against* it."
    )
    raw_json = {l: round(float(p), 4) for l, p in ranked}
    rank_df = pd.DataFrame(
        [{"Rank": i + 1, "Emotion": l, "Probability": round(p, 3)} for i, (l, p) in enumerate(shown)]
    )
    rank_table_html = build_rank_table_html(rank_df)

    new_history = history + [{"text": text[:70], "label": target_label, "pct": round(shown[0][1] * 100)}]

    progress(1.0, desc="Done")
    return (spotlight_html, other_bars_html, why_html, comparison_html, mismatch_html,
            lime_highlight, lime_chart, ig_highlight, ig_chart,
            comp_chart, suff_chart, note, raw_json, rank_table_html,
            build_history_html(new_history), new_history)


def clear_analyze(history):
    empty_html = build_rank_table_html(pd.DataFrame(columns=["Rank", "Emotion", "Probability"]))
    return (
        "",              
        "",              
        "",              
        "",              
        "",              
        "",              
        [],              
        None,            
        [],              
        None,            
        None,            
        None,            
        "Type a sentence and click **Analyze**.", 
        {},             
        empty_html,     
        build_history_html(history), 
        history,         
    )


def toggle_view(mode):
    detailed = (mode == "Detailed")
    return gr.update(visible=detailed), gr.update(visible=detailed)


CAROUSEL_SLIDES = [
    ("01 · Input", "A raw sentence is entered by the user — e.g. *\"I am really happy today.\"*"),
    ("02 · Tokenizer", "The sentence is broken into subword tokens DistilBERT can process."),
    ("03 · DistilBERT", "The encoder produces contextual representations of the tokens."),
    ("04 · Prediction", "A sigmoid layer outputs an independent probability for each of the 28 emotions."),
    ("05 · Explanation", "LIME and Integrated Gradients each identify which words most influenced the top prediction."),
    ("06 · Trust check", "Comprehensiveness and sufficiency measure whether those words actually matter to the model."),
]


def render_slide(idx):
    title, body = CAROUSEL_SLIDES[idx]
    return f'<div class="slide"><p class="slide-num">{title}</p><p class="slide-body">{body}</p></div>'


def carousel_prev(idx):
    idx = (idx - 1) % len(CAROUSEL_SLIDES)
    return render_slide(idx), idx


def carousel_next(idx):
    idx = (idx + 1) % len(CAROUSEL_SLIDES)
    return render_slide(idx), idx


def make_label_distribution_chart():
    data_path = TRAIN_META.get("data_path", "go_emotions_dataset.csv")
    if not os.path.exists(data_path):
        return None
    df = pd.read_csv(data_path)
    if "example_very_unclear" in df.columns:
        df = df[df["example_very_unclear"] == False]  
    counts = df[EMOTION_COLUMNS].sum().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(11, 4.5))
    fig.patch.set_alpha(0.0)
    ax.set_facecolor("none")
    ax.bar(counts.index, counts.values, color="#818CF8")
    ax.set_xticklabels(counts.index, rotation=75, ha="right", color="#CBD5E1")
    ax.set_ylabel("Number of examples", color="#CBD5E1")
    ax.set_title("Label distribution (class imbalance motivating pos_weight)",
                 fontweight="bold", color="#E2E8F0")
    ax.tick_params(colors="#CBD5E1")
    for spine in ax.spines.values():
        spine.set_color("#334155")
    fig.tight_layout()
    return fig


def make_pipeline_diagram():
    stages = ["Input\nsentence", "Tokenizer", "DistilBERT\nencoder",
              "28 emotion\nprobabilities", "LIME / IG\nexplainers", "Highlighted\nexplanation"]
    fig, ax = plt.subplots(figsize=(12, 2.2))
    fig.patch.set_alpha(0.0)
    ax.set_xlim(0, len(stages))
    ax.set_ylim(0, 1)
    ax.axis("off")
    for i, s in enumerate(stages):
        ax.add_patch(plt.Rectangle((i + 0.05, 0.25), 0.9, 0.5,
                                    facecolor="#1E1B3A", edgecolor="#818CF8", linewidth=1.5))
        ax.text(i + 0.5, 0.5, s, ha="center", va="center", fontsize=9, color="#E2E8F0")
        if i < len(stages) - 1:
            ax.annotate("", xy=(i + 1.0, 0.5), xytext=(i + 0.95, 0.5),
                        arrowprops=dict(arrowstyle="->", lw=1.5, color="#818CF8"))
    fig.tight_layout()
    return fig


def load_eval_image(filename):
    path = os.path.join(EVAL_DIR, filename)
    return path if os.path.exists(path) else None


def load_eval_plot(filename):
    path = os.path.join(EVAL_DIR, filename)
    if not os.path.exists(path):
        return None
    try:
        return pio.read_json(path)
    except Exception as e:
        print(f"Failed to load {path}: {e}")
        return None


def build_limitations_markdown():
    lines = [
        "- **Sarcasm/irony** are not reliably detected — the model reads text literally.",
        "- **Overlapping emotions** (e.g. \"disappointment\" vs \"sadness\") are hard even for "
        "human annotators, which caps achievable F1.",
        "- **LIME and Integrated Gradients are approximations**, not ground truth — they can "
        "disagree, which is what the faithfulness metrics measure.",
        "- **CPU latency**: LIME re-queries the model many times per explanation and is "
        "noticeably slower without a GPU.",
        "- **Word-highlight matching**: attribution scores are computed on subword tokens, "
        "then re-merged onto whole words in the original sentence. Punctuation-heavy or "
        "contracted words occasionally fail to match — when this happens the app now shows "
        "an explicit note rather than silently skipping the highlight.",
    ]
    if "full_dataset_size" in TRAIN_META:
        lines.append(
            f"- **Subsampled training data**: trained on {TRAIN_META['sample_size']:,} of "
            f"{TRAIN_META['full_dataset_size']:,} available rows "
            f"({TRAIN_META.get('sample_fraction', 0):.1%}) as a compute tradeoff — metrics "
            f"reflect this smaller sample, not the full dataset."
        )
    if "token_length_percentiles" in TRAIN_META:
        p95 = TRAIN_META["token_length_percentiles"].get("95")
        p95 = p95 if p95 is not None else TRAIN_META["token_length_percentiles"].get(95)
        lines.append(
            f"- **Sequence length ({MAX_LEN} tokens)**: chosen from the 95th percentile of "
            f"observed token lengths (p95 ≈ {p95}) rather than an arbitrary guess."
        )
    return "\n".join(lines)


custom_theme = gr.themes.Base(
    primary_hue=gr.themes.colors.violet,
    secondary_hue=gr.themes.colors.indigo,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    body_background_fill="#0B0E1A",
    body_text_color="#E2E8F0",
    background_fill_primary="#12162A",
    background_fill_secondary="#161B30",
    border_color_primary="#262B45",
    button_primary_background_fill="linear-gradient(90deg, #7C3AED 0%, #4F46E5 100%)",
    button_primary_background_fill_hover="linear-gradient(90deg, #6D28D9 0%, #4338CA 100%)",
    button_primary_text_color="white",
    block_radius="20px",
    block_label_text_color="#CBD5E1",
    input_background_fill="#0F1224",
)

CUSTOM_CSS = """
html, body, .gradio-container {
    background:
        radial-gradient(circle at 10% 5%, rgba(124,58,237,0.18), transparent 30%),
        radial-gradient(circle at 90% 10%, rgba(79,70,229,0.16), transparent 30%),
        radial-gradient(circle at 50% 100%, rgba(192,38,211,0.10), transparent 35%),
        #0B0E1A !important;
}
.hero-banner {
    background: linear-gradient(135deg, rgba(79,70,229,0.35) 0%, rgba(124,58,237,0.35) 55%, rgba(192,38,211,0.30) 100%);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 24px; padding: 40px 34px; margin-bottom: 20px;
    backdrop-filter: blur(18px); -webkit-backdrop-filter: blur(18px);
    position: relative; overflow: hidden;
    animation: fadein 0.6s ease;
}
@keyframes fadein { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }
.hero-eyebrow { color: #C4B5FD !important; font-size: 0.78em !important; font-weight: 700 !important;
    letter-spacing: 0.14em !important; text-transform: uppercase; margin: 0 0 8px 0 !important; }
.hero-banner h1 { color: #ffffff !important; font-size: 2.1em !important; margin: 0 0 8px 0 !important;
    font-weight: 800 !important; letter-spacing: -0.02em; }
.hero-sub { color: #DDD6FE !important; font-size: 1.0em !important; margin: 0 0 8px 0 !important; }
.hero-banner p.hero-tagline { color: #F1F5F9 !important; font-size: 1.15em !important; margin: 6px 0 0 0 !important; }
.badge-row { margin-top: 18px; display: flex; flex-wrap: wrap; gap: 8px; }
.badge { display: inline-block; background: rgba(255,255,255,0.10); color: #EDE9FE;
    border: 1px solid rgba(255,255,255,0.18); padding: 6px 16px; border-radius: 999px;
    font-size: 0.82em; font-weight: 600; }
.stats-row { display: flex; gap: 14px; flex-wrap: wrap; margin-bottom: 20px; }
.stat-card { flex: 1; min-width: 130px; background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.08); border-radius: 16px; padding: 16px; text-align: center;
    backdrop-filter: blur(10px); }
.stat-num { font-size: 1.6em; font-weight: 800; color: #A78BFA; }
.stat-label { font-size: 0.82em; color: #94A3B8; margin-top: 4px; }
.section-card { background: rgba(255,255,255,0.045) !important; border-radius: 18px !important;
    padding: 20px !important; border: 1px solid rgba(255,255,255,0.08) !important;
    backdrop-filter: blur(14px); -webkit-backdrop-filter: blur(14px);
    transition: transform 0.2s ease, box-shadow 0.2s ease; }
.section-card:hover { transform: translateY(-3px); box-shadow: 0 16px 34px rgba(0,0,0,0.35) !important; }
.section-eyebrow { color: #A78BFA !important; font-size: 0.74em !important; font-weight: 700 !important;
    letter-spacing: 0.1em !important; text-transform: uppercase; margin: 0 0 2px 0 !important; }
.section-title { font-size: 1.15em !important; font-weight: 700 !important; color: #F1F5F9 !important; margin: 0 0 10px 0 !important; }
.info-box { background: rgba(124,58,237,0.12); border-left: 4px solid #A78BFA; border-radius: 10px;
    padding: 12px 18px !important; font-size: 0.98em; color: #E2E8F0; }
.mini-label { color: #94A3B8; font-size: 0.78em; text-transform: uppercase; letter-spacing: 0.08em;
    font-weight: 700; margin: 0 0 8px 0; }
.muted-text { color: #64748B; font-size: 0.9em; }
.mismatch-note { background: rgba(251,191,36,0.10); border-left: 3px solid #FBBF24; border-radius: 8px;
    padding: 8px 12px; color: #FDE68A; font-size: 0.85em; margin: 8px 0 0 0; }
.legend-footnote { margin-top: 6px; }
.emo-bars { display: flex; flex-direction: column; gap: 14px; }
.emo-row-top { display: flex; justify-content: space-between; font-size: 0.92em; margin-bottom: 4px; color: #E2E8F0; }
.emo-name { font-weight: 600; text-transform: capitalize; }
.emo-pct { font-weight: 700; color: #A78BFA; }
.emo-bar-track { background: rgba(255,255,255,0.08); border-radius: 999px; height: 10px; overflow: hidden; }
.emo-bar-fill { background: linear-gradient(90deg, #7C3AED, #4F46E5); height: 100%; border-radius: 999px; }
.spotlight { text-align: center; padding: 10px 0; }
.spotlight-icon { font-size: 2.6em; margin-bottom: 4px; }
.spotlight-label { font-size: 1.5em; font-weight: 800; letter-spacing: 0.04em; color: #F1F5F9; }
.spotlight-pct { font-size: 2.2em; font-weight: 800; color: #A78BFA; margin: 4px 0; }
.spotlight-bar-track { background: rgba(255,255,255,0.08); border-radius: 999px; height: 12px;
    overflow: hidden; max-width: 320px; margin: 8px auto 0; }
.spotlight-bar-fill { background: linear-gradient(90deg, #7C3AED, #C026D3); height: 100%; border-radius: 999px; }
.why-card { padding: 4px 2px; }
.why-text { color: #CBD5E1; font-size: 0.98em; margin-top: 4px; }
.legend-row { display: flex; gap: 10px; margin-bottom: 6px; flex-wrap: wrap; }
.legend-chip { padding: 4px 12px; border-radius: 999px; font-size: 0.82em; font-weight: 600; }
.legend-pos { background: rgba(52,211,153,0.15); color: #6EE7B7; border: 1px solid rgba(52,211,153,0.35); }
.legend-neg { background: rgba(248,113,113,0.15); color: #FCA5A5; border: 1px solid rgba(248,113,113,0.35); }
.compare-grid { display: flex; gap: 20px; margin-bottom: 10px; }
.compare-col { flex: 1; min-width: 0; }
.word-list { list-style: none; padding: 0; margin: 4px 0; }
.word-list li { background: rgba(255,255,255,0.06); border-radius: 8px; padding: 6px 10px;
    margin-bottom: 6px; color: #E2E8F0; font-size: 0.92em; }
.agreement-row { display: flex; align-items: center; gap: 12px; margin: 10px 0 4px; }
.agreement-bar-track { flex: 1; background: rgba(255,255,255,0.08); border-radius: 999px; height: 10px; overflow: hidden; }
.agreement-bar-fill { background: linear-gradient(90deg, #FBBF24, #818CF8); height: 100%; border-radius: 999px; }
.agreement-pct { font-weight: 700; color: #F1F5F9; min-width: 42px; text-align: right; }
.summary-text { color: #94A3B8; font-size: 0.9em; margin-top: 4px; }
.history-item { border-bottom: 1px solid rgba(255,255,255,0.06); padding: 8px 0; }
.history-top { color: #E2E8F0; font-size: 0.92em; }
.history-text { color: #64748B; font-size: 0.82em; font-style: italic; }
.slide { text-align: center; padding: 24px 10px; min-height: 90px; }
.slide-num { color: #A78BFA; font-weight: 800; letter-spacing: 0.06em; margin-bottom: 8px; }
.slide-body { color: #E2E8F0; font-size: 1.05em; }
.app-footer { text-align: center; color: #64748B; font-size: 0.85em; padding: 24px 0 8px; }
.app-footer strong { color: #CBD5E1; }
footer { display: none !important; }

.img-frame {
    position: relative; border-radius: 14px; overflow: hidden;
    background: linear-gradient(90deg, #161B30 25%, #1E2440 37%, #161B30 63%);
    background-size: 400% 100%;
    animation: shimmer 1.6s ease-in-out infinite;
}
.img-frame img { position: relative; z-index: 1; border-radius: 14px; display: block; }
@keyframes shimmer { 0% { background-position: 100% 50%; } 100% { background-position: 0% 50%; } }

/* NEW — replaces the fragile gr.Dataframe CSS overrides. Plain HTML table,
   colors fully controlled here, so it renders correctly on every Gradio
   version regardless of internal component class changes. */
.rank-table { width: 100%; border-collapse: collapse; }
.rank-table th, .rank-table td {
    padding: 8px 12px; text-align: left; color: #E2E8F0 !important;
    border-bottom: 1px solid rgba(255,255,255,0.08);
}
.rank-table thead th {
    color: #C4B5FD !important; font-weight: 700; background: rgba(255,255,255,0.04);
}
.rank-table tbody tr:hover td { background: rgba(124,58,237,0.10); }

/* -------- FIX: gr.Examples table text was invisible on dark theme -------- */
.gradio-container .gr-samples-table,
.gradio-container [id*="component"] table,
.gradio-container table {
    background: #12162A !important;
    color: #E2E8F0 !important;
    border-color: #262B45 !important;
}
.gradio-container table td,
.gradio-container table th,
.gradio-container [class*="dataset"] td,
.gradio-container [class*="dataset"] span,
.gradio-container [class*="dataset"] p {
    color: #E2E8F0 !important;
    background: transparent !important;
}
.gradio-container table tr:hover td {
    background: rgba(124,58,237,0.15) !important;
}

/* gr.JSON contrast fix (raw prediction probabilities panel) */
.gradio-container .json-holder,
.gradio-container .json-holder *,
.gradio-container [class*="json"] {
    background: #12162A !important;
    color: #E2E8F0 !important;
}
.gradio-container .json-holder pre,
.gradio-container .json-holder code,
.gradio-container [class*="json"] pre,
.gradio-container [class*="json"] code {
    background: #12162A !important;
    color: #E2E8F0 !important;
}
.gradio-container .token.string  { color: #93C5FD !important; }
.gradio-container .token.number  { color: #FBBF24 !important; }
.gradio-container .token.boolean { color: #34D399 !important; }
.gradio-container .token.key,
.gradio-container .token.property { color: #C4B5FD !important; }
.gradio-container .token.punctuation { color: #94A3B8 !important; }
.gradio-container .cm-editor, .gradio-container .cm-content, .gradio-container .cm-line {
    background: #12162A !important;
    color: #E2E8F0 !important;
}

@media (max-width: 768px) {
    .hero-banner { padding: 24px 18px; border-radius: 18px; }
    .hero-banner h1 { font-size: 1.5em !important; }
    .hero-sub, .hero-banner p.hero-tagline { font-size: 0.92em !important; }
    .badge { padding: 5px 12px; font-size: 0.76em; }
    .stats-row { flex-direction: column; }
    .stat-card { min-width: 0; }
    .compare-grid { flex-direction: column; gap: 12px; }
    .section-card { padding: 14px !important; border-radius: 14px !important; }
    .spotlight-label { font-size: 1.2em; }
    .spotlight-pct { font-size: 1.7em; }
}
"""


def section_header(eyebrow, title):
    return f'<p class="section-eyebrow">{eyebrow}</p><p class="section-title">{title}</p>'


with gr.Blocks(title="Explainable NLP — Emotion Classifier", theme=custom_theme, css=CUSTOM_CSS) as demo:
    history_state = gr.State([])
    carousel_idx = gr.State(0)

    gr.HTML(
        """
        <div class="hero-banner">
          <p class="hero-eyebrow">🧠 Explainable NLP</p>
          <h1>Emotion Intelligence Studio</h1>
          <p class="hero-sub">Explainable NLP for transparent multi-label emotion classification.</p>
          <p class="hero-tagline">Understand not just what the model predicts — but why.</p>
          <div class="badge-row">
            <span class="badge">🤖 DistilBERT</span>
            <span class="badge">🔍 LIME</span>
            <span class="badge">🧠 Integrated Gradients</span>
            <span class="badge">🛡️ Trust Checked</span>
          </div>
        </div>
        """
    )
    gr.HTML(STATS_HTML)

    with gr.Tabs():
        with gr.Tab("✨ Analyze"):
            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Live Demo", "🔎 Analyze a Sentence"))
                text_input = gr.Textbox(
                    label="Your sentence",
                    placeholder="e.g. Thank you so much for helping me today",
                    lines=2,
                )
                with gr.Row():
                    threshold_slider = gr.Slider(0.05, 0.9, value=0.3, step=0.05, label="Probability threshold")
                    top_n_slider = gr.Slider(1, 10, value=5, step=1, label="Max labels to show")
                with gr.Row():
                    submit_btn = gr.Button("✨ Analyze", variant="primary", scale=2)
                    clear_btn = gr.Button("🗑️ Clear", scale=1)
                gr.Examples(
                    examples=[
                        "Thank you so much for helping me today, I really appreciate it.",
                        "I am so angry that they cancelled the event without telling anyone.",
                        "I'm not sure how I feel about this, it's confusing.",
                    ],
                    inputs=text_input,
                    label="Quick examples",
                )
                view_radio = gr.Radio(
                    ["Simple", "Detailed"], value="Simple", label="View",
                    info="Simple hides the explainer comparison and faithfulness charts below to reduce clutter.",
                )

            note_box = gr.Markdown(elem_classes=["info-box"])

            with gr.Row():
                with gr.Column(scale=1):
                    with gr.Group(elem_classes=["section-card"]):
                        gr.HTML(section_header("Model Output", "🎯 Prediction Spotlight"))
                        spotlight_out = gr.HTML()
                with gr.Column(scale=1):
                    with gr.Group(elem_classes=["section-card"]):
                        gr.HTML(section_header("Model Output", "📋 Other Detected Emotions"))
                        other_bars_out = gr.HTML()

            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Plain-language Summary", "❓ Why This Emotion?"))
                why_out = gr.HTML()

            gr.HTML(LEGEND_HTML)
            mismatch_note_out = gr.HTML()

            with gr.Row():
                with gr.Column():
                    with gr.Group(elem_classes=["section-card"]):
                        gr.HTML(section_header("Explainability · Method 1", "🔍 LIME"))
                        lime_highlight_out = gr.HighlightedText(label="Highlighted words", show_legend=True)
                        lime_chart_out = gr.Plot(label="Top word weights")
                with gr.Column():
                    with gr.Group(elem_classes=["section-card"]):
                        gr.HTML(section_header("Explainability · Method 2", "🧠 Integrated Gradients"))
                        ig_highlight_out = gr.HighlightedText(label="Highlighted words", show_legend=True)
                        ig_chart_out = gr.Plot(label="Top word weights")

            with gr.Group(elem_classes=["section-card"], visible=False) as comparison_group:
                gr.HTML(section_header("Explanation Comparison", "🔬 LIME vs Integrated Gradients"))
                comparison_out = gr.HTML()

            with gr.Group(elem_classes=["section-card"], visible=False) as faithfulness_group:
                gr.HTML(section_header("Trust Check", "🛡️ Faithfulness"))
                with gr.Row():
                    comp_chart_out = gr.Plot(label="Comprehensiveness")
                    suff_chart_out = gr.Plot(label="Sufficiency drop")

            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("This Session", "🕓 Recent Analyses"))
                history_out = gr.HTML()

            with gr.Accordion("Advanced technical details", open=False):
                rank_table_out = gr.HTML()   # CHANGED — was gr.Dataframe(...)
                raw_json_out = gr.JSON(label="Raw prediction probabilities")

            view_radio.change(fn=toggle_view, inputs=[view_radio], outputs=[comparison_group, faithfulness_group])

            outputs_list = [spotlight_out, other_bars_out, why_out, comparison_out, mismatch_note_out,
                             lime_highlight_out, lime_chart_out, ig_highlight_out, ig_chart_out,
                             comp_chart_out, suff_chart_out, note_box, raw_json_out, rank_table_out,
                             history_out, history_state]

            submit_btn.click(fn=analyze, inputs=[text_input, threshold_slider, top_n_slider, history_state],
                              outputs=outputs_list)
            text_input.submit(fn=analyze, inputs=[text_input, threshold_slider, top_n_slider, history_state],
                               outputs=outputs_list)
            clear_btn.click(fn=clear_analyze, inputs=[history_state],
                             outputs=[text_input] + outputs_list)

        with gr.Tab("📊 Evaluate"):
            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Benchmark", "📈 Model Performance"))
                with gr.Row():
                    gr.Number(value=round(TRAIN_META["micro_f1"], 3), label="Micro F1", interactive=False)
                    gr.Number(value=round(TRAIN_META["macro_f1"], 3), label="Macro F1", interactive=False)
                    gr.Number(value=round(TRAIN_META["precision"], 3), label="Precision", interactive=False)
                    gr.Number(value=round(TRAIN_META["recall"], 3), label="Recall", interactive=False)

            per_label_fig = load_eval_plot("per_label_f1.json")
            if per_label_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Per-class Breakdown", "🏷️ Per-label F1"))
                    gr.Plot(value=per_label_fig, show_label=False)

            cm_fig = load_eval_plot("confusion_matrices.json")
            if cm_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Error Analysis", "🧮 Confusion Matrices — Top 8 Emotions"))
                    gr.Plot(value=cm_fig, show_label=False)

            roc_fig = load_eval_plot("roc_curves.json")
            pr_fig = load_eval_plot("pr_curves.json")
            if roc_fig or pr_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Threshold Behavior", "📉 ROC & Precision–Recall"))
                    with gr.Row():
                        if roc_fig:
                            with gr.Column():
                                gr.Markdown("**ROC Curves**")
                                gr.Plot(value=roc_fig, show_label=False)
                        if pr_fig:
                            with gr.Column():
                                gr.Markdown("**Precision–Recall Curves**")
                                gr.Plot(value=pr_fig, show_label=False)

            freq_fig = load_eval_plot("true_vs_pred_freq.json")
            if freq_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Calibration", "⚖️ True vs. Predicted Frequency"))
                    gr.Plot(value=freq_fig, show_label=False)

            faith_fig = load_eval_plot("faithfulness_comparison.json")
            if faith_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Trust Check · Batch Average", "🛡️ Faithfulness — LIME vs IG"))
                    gr.Plot(value=faith_fig, show_label=False)

            if not any([per_label_fig, cm_fig, roc_fig, pr_fig, freq_fig, faith_fig]):
                gr.Markdown("*Run `evaluate.py` after training to populate this tab.*")

        with gr.Tab("🧬 How It Works"):
            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Overview", "⚙️ Objective"))
                gr.Markdown(
                    """
Fine-tune DistilBERT for multi-label emotion classification on GoEmotions (28 labels;
a sentence can carry more than one emotion), then explain individual predictions with
two independent methods — **LIME** and **Integrated Gradients** — and quantify how much
to trust each explanation using faithfulness metrics (comprehensiveness, sufficiency).
"""
                )
                gr.Plot(value=make_pipeline_diagram(), show_label=False)

            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Walkthrough", "🎬 How Explainable NLP Works"))
                slide_html = gr.HTML(render_slide(0))
                with gr.Row():
                    prev_btn = gr.Button("◀ Previous")
                    next_btn = gr.Button("Next ▶")
                prev_btn.click(fn=carousel_prev, inputs=[carousel_idx], outputs=[slide_html, carousel_idx])
                next_btn.click(fn=carousel_next, inputs=[carousel_idx], outputs=[slide_html, carousel_idx])

            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Under the Hood", "🤖 Model Information"))
                gr.Markdown(
                    f"""
| | |
|---|---|
| **Model** | `distilbert-base-uncased` (fine-tuned) |
| **Dataset** | GoEmotions |
| **Labels** | 28 emotions |
| **Explainers** | LIME + Integrated Gradients (Captum) |
| **Interface** | Gradio |
| **Inference device** | {DEVICE.upper()} |
"""
                )

            dist_fig = make_label_distribution_chart()
            if dist_fig is not None:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Data", "📊 Label Distribution"))
                    gr.Plot(value=dist_fig, show_label=False)

            token_hist_fig = load_eval_plot("token_length_hist.json")
            if token_hist_fig:
                with gr.Group(elem_classes=["section-card"]):
                    gr.HTML(section_header("Design Decision", "📏 Token Length Distribution"))
                    gr.Plot(value=token_hist_fig, show_label=False)
            else:
                token_hist_img = load_eval_image("token_length_hist.png")
                if token_hist_img:
                    with gr.Group(elem_classes=["section-card"]):
                        gr.HTML(section_header("Design Decision", "📏 Token Length Distribution"))
                        gr.Image(value=token_hist_img, show_label=False, elem_classes=["img-frame"])

            with gr.Group(elem_classes=["section-card"]):
                gr.HTML(section_header("Reference", "📚 Design Decisions & Limitations"))
                with gr.Accordion("🤖 Why DistilBERT?", open=False):
                    gr.Markdown(
                        "DistilBERT retains ~97% of BERT's language understanding at roughly "
                        "60% of the size and inference cost, making it practical to fine-tune "
                        "and serve without a dedicated GPU cluster."
                    )
                with gr.Accordion("🔍 Why LIME and Integrated Gradients?", open=False):
                    gr.Markdown(
                        "LIME perturbs the input and fits a local surrogate model; Integrated "
                        "Gradients traces gradients along a path from a baseline to the actual "
                        "input. They are fundamentally different explanation families — using "
                        "both, and checking where they agree or disagree, gives a more "
                        "trustworthy picture than relying on either alone."
                    )
                with gr.Accordion("🛡️ What is faithfulness?", open=False):
                    gr.Markdown(
                        "**Comprehensiveness** asks: if we remove the words an explainer says "
                        "matter most, does the predicted probability actually drop? "
                        "**Sufficiency** asks the opposite: if we keep *only* those words, is the "
                        "prediction preserved? Together they measure whether an explanation "
                        "reflects what the model actually used."
                    )
                with gr.Accordion("⚖️ Why pos_weight?", open=False):
                    gr.Markdown(
                        "Neutral appears in roughly a third of examples; some emotions "
                        "(e.g. gratitude) appear in under 2%. Reweighting the loss per label "
                        "prevents the model from ignoring rare labels."
                    )
                with gr.Accordion("⚠️ Limitations", open=False):
                    gr.Markdown(build_limitations_markdown())

    gr.HTML(
        """
        <div class="app-footer">
          <strong>Explainable NLP — Emotion Intelligence Studio</strong><br>
          DistilBERT • GoEmotions • LIME • Integrated Gradients
        </div>
        """
    )

if __name__ == "__main__":
    demo.queue()
    demo.launch()

Loading model...


C:\Users\Mahalakshmi\AppData\Local\Temp\ipykernel_26328\3990772553.py:496: UserWarning:

set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.



Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
